# Module A2 — Eye Closure Detection  v3

## What changed from v2
| What | v2 | v3 |
|---|---|---|
| Crop extractor | MediaPipe 68-pt landmarks | **YuNet 5-pt IOD-based** (matches inference exactly) |
| Crop unit | 112×224 combined strip | **64×64 per-eye** (L + R separately) |
| Datasets | NTHU-DDD only | **NTHU + MRL Eye + dhirdevansh eye dataset** |
| Loss | BCE + pos_weight | **Focal Loss γ=2.0 + balanced sampler** |
| Attention | None | **CBAM** between backbone and head |
| Regularisation | Cosine LR | Cosine LR + **SWA** from epoch 17 |
| Lighting normalisation | None | **CLAHE / gamma** pre-crop + augmentation |
| Threshold | 0.5 hardcoded | **Grid search 0.01–0.99**, deploy at best val-F1 |

**Root cause of v2 failure:** trained on MediaPipe 68-pt crops, inferred on YuNet 5-pt crops.  
Different geometry → distribution mismatch → systematic over-prediction of closure.  
v3 extracts ALL training crops with the exact same YuNet IOD formula used in the app.

## Required Kaggle datasets (add all 4 in Data tab before running)
1. `suyashpokle2/nthu-ddd` — NTHU-DDD drowsy driver video frames  
2. `suyashpokle2/manifests-cv-kaggle-fullpaths` — fold manifests  
3. `tauilabdelilah/mrl-eye-dataset` — 84,898 IR open/closed eye images  
   ⚠️ MRL filenames encode labels in position [4] of the underscore-split name  
   e.g. `s0001_02694_0_0_**1**_0_0_01.png` → position 4 = 1 = OPEN  
4. `dhirdevansh/eye-dataset-openclose-for-drowsiness-prediction` — replaces CEW  
   (folder-based labels: `open/` and `closed/` subfolders)  

## Output (after training completes)
- `a2_fold_1_best.pt` through `a2_fold_4_best.pt`  
- `a2_ensemble_meta_v3.json` — thresholds, AUC, fold selection  

Upload all to `@DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS` on Snowflake.


In [25]:
# ── Install / upgrade packages ────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("albumentations>=1.3.1", "timm>=0.9.2", "opencv-python-headless>=4.8")
print("Packages ready.")

Packages ready.


In [26]:
# ── Imports ───────────────────────────────────────────────────────────────────
import gc, json, math, os, random, re, shutil, urllib.request, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

import timm
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings("ignore")
print("torch:", torch.__version__, "| timm:", timm.__version__)

torch: 2.10.0+cu128 | timm: 1.0.25


In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
@dataclass
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────────
    nthu_data_dir:    str = "/kaggle/input/datasets/suyashpokle2/nthu-ddd/NTHU DDD"
    manifest_root:    str = "/kaggle/input/datasets/suyashpokle2/manifests-cv-kaggle-fullpaths/manifests_cv_kaggle_fullpaths"
    mrl_dir:          str = "/kaggle/input/datasets/tauilabdelilah/mrl-eye-dataset"       # tauilabdelilah/mrl-eye-dataset
    ext_dir:          str = "/kaggle/input/datasets/dhirdevansh/eye-dataset-openclose-for-drowsiness-prediction"  # dhirdevansh dataset (replaces CEW)
    output_root:      str = "/kaggle/working/model_a2_v3"
    yunet_model_path: str = "/kaggle/working/face_detection_yunet_2023mar.onnx"
    crop_cache_root:  str = "/kaggle/working/eye_crop_cache_v3"

    # ── Which folds to run ─────────────────────────────────────────────────────
    fold_ids: List[int] = field(default_factory=lambda: [1, 2, 3, 4])

    # ── Model ──────────────────────────────────────────────────────────────────
    backbone:  str   = "mobilenetv3_small_100"   # keeps fast inference on CPU
    img_size:  int   = 64                         # 64×64 per-eye crop
    use_cbam:  bool  = True
    dropout:   float = 0.30

    # ── Training ───────────────────────────────────────────────────────────────
    seed:         int   = 42
    epochs:       int   = 25
    batch_size:   int   = 64
    lr:           float = 1e-4
    weight_decay: float = 1e-4
    grad_clip:    float = 1.0
    num_workers:  int   = 2
    use_amp:      bool  = True
    deterministic: bool = True
    cudnn_benchmark: bool = False

    # ── Loss ───────────────────────────────────────────────────────────────────
    focal_gamma:    float = 2.0    # Focal Loss: downweights easy open-eye frames
    focal_alpha:    float = 0.75   # weight on positive (closed) class
    use_focal_loss: bool  = True

    # ── Balanced sampler ───────────────────────────────────────────────────────
    use_balanced_sampler: bool = True

    # ── SWA ────────────────────────────────────────────────────────────────────
    use_swa:    bool  = True
    swa_start:  float = 0.35       # fraction of epochs before SWA activates
    swa_lr:     float = 5e-5

    # ── Threshold search ───────────────────────────────────────────────────────
    threshold_min:  float = 0.01
    threshold_max:  float = 0.99
    threshold_steps: int  = 199

    # ── Early stopping ─────────────────────────────────────────────────────────
    patience: int   = 8
    min_epochs: int = 12

    # ── Crop extraction ────────────────────────────────────────────────────────
    iod_width_ratio:  float = 0.70   # crop_width  = IOD * iod_width_ratio
    iod_height_ratio: float = 0.50   # crop_height = IOD * iod_height_ratio
    rebuild_crop_cache: bool = False
    save_cache_every:   int  = 2000

    # ── Dataset mixing ─────────────────────────────────────────────────────────
    # How many MRL + CEW samples to include per training fold (0 = disabled)
    mrl_samples_per_fold:  int = 8000   # ~4k open + ~4k closed from MRL (label from filename[4])
    ext_samples_per_fold:  int = 2000   # extra samples from dhirdevansh eye dataset

    # ── Validation ─────────────────────────────────────────────────────────────
    min_known_samples_per_split: int = 300

CFG = CFG()
print("CFG loaded.  epochs:", CFG.epochs, "| img_size:", CFG.img_size,
      "| backbone:", CFG.backbone)

CFG loaded.  epochs: 25 | img_size: 64 | backbone: mobilenetv3_small_100


In [28]:
# ══════════════════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════════════════
def seed_everything(seed=42, deterministic=True, cudnn_benchmark=False):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = cudnn_benchmark if not deterministic else False

def seed_worker(worker_id):
    ws = torch.initial_seed() % (2**32)
    np.random.seed(ws); random.seed(ws)

def ensure_dir(p): os.makedirs(p, exist_ok=True)

def compact_text(*parts):
    text = " ".join(str(p) for p in parts if p is not None)
    return re.sub(r"[^a-z0-9]+", "", text.lower())

def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

def safe_auc(y_true, y_prob):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2: return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def safe_ap(y_true, y_prob):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2: return float("nan")
    return float(average_precision_score(y_true, y_prob))

def find_best_threshold(y_true, y_prob, n_steps=199):
    """Grid search for threshold maximising F1 on the provided split."""
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    best_f1, best_thr = -1.0, 0.5
    for thr in np.linspace(CFG.threshold_min, CFG.threshold_max, n_steps):
        preds = (y_prob >= thr).astype(int)
        if preds.sum() == 0: continue
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1: best_f1, best_thr = f1, float(thr)
    return best_thr, best_f1

def normalize_manifest_path(path_value, data_dir):
    if pd.isna(path_value): return ""
    p = str(path_value).strip().replace("\\", "/")
    if os.path.isabs(p) and os.path.exists(p): return p
    joined = os.path.join(data_dir, p)
    if os.path.exists(joined): return joined
    parts = [x for x in p.split("/") if x]
    for idx in range(len(parts)):
        c = os.path.join(data_dir, *parts[idx:])
        if os.path.exists(c): return c
    return joined

print("Utilities ready.")

Utilities ready.


In [29]:
# ══════════════════════════════════════════════════════════════════════════════
# LIGHTING NORMALISATION
# Matches the exact logic in streamlit_app_v7.py  normalize_lighting()
# Applied BEFORE eye crop extraction so training crops match inference crops.
# ══════════════════════════════════════════════════════════════════════════════
def normalize_lighting(frame_bgr: np.ndarray) -> Tuple[np.ndarray, str]:
    """
    Returns (normalised_frame, method_label).
    Mirrors streamlit_app_v7.py normalize_lighting() exactly.
    method_label: 'normal' | 'overexposed_correction' | 'underexposed_correction'
                  | 'low_contrast_correction' | 'error'
    """
    if frame_bgr is None: return frame_bgr, "none"
    try:
        gray   = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        mean_b = float(np.mean(gray))
        std_b  = float(np.std(gray))

        if mean_b > 175:                             # glare / overexposure
            gamma = 1.8
            lut   = np.array([((i/255.0)**(1.0/gamma))*255
                              for i in range(256)], dtype=np.uint8)
            out   = cv2.LUT(frame_bgr, lut)
            hsv   = cv2.cvtColor(out, cv2.COLOR_BGR2HSV)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            hsv[:, :, 2] = clahe.apply(hsv[:, :, 2])
            return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR), "overexposed_correction"

        elif mean_b < 55:                            # dark / underexposed
            lab   = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
            clahe = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8, 8))
            lab[:, :, 0] = clahe.apply(lab[:, :, 0])
            return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), "underexposed_correction"

        elif std_b < 22:                             # low contrast
            lab   = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            lab[:, :, 0] = clahe.apply(lab[:, :, 0])
            return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), "low_contrast_correction"

        return frame_bgr, "normal"
    except Exception:
        return frame_bgr, "error"

print("normalize_lighting() defined.")

normalize_lighting() defined.


In [30]:
# ══════════════════════════════════════════════════════════════════════════════
# YUNET FACE DETECTOR
# Matches streamlit_app_v7.py  detect_face_eyes() / _yunet_detect()
# ══════════════════════════════════════════════════════════════════════════════

def download_yunet(dst: str):
    if os.path.exists(dst):
        print("YuNet already present:", dst)
        return dst
    url = ("https://github.com/opencv/opencv_zoo/raw/main/"
           "models/face_detection_yunet/face_detection_yunet_2023mar.onnx")
    print("Downloading YuNet …")
    urllib.request.urlretrieve(url, dst)
    print("Downloaded YuNet →", dst)
    return dst


def make_yunet_detector(model_path: str) -> cv2.FaceDetectorYN:
    return cv2.FaceDetectorYN.create(
        model=model_path, config="", input_size=(320, 320),
        score_threshold=0.50,   # slightly lower for training to capture more faces
        nms_threshold=0.30, top_k=1)


def yunet_detect_eyes(frame_bgr: np.ndarray,
                       detector: cv2.FaceDetectorYN
                       ) -> Optional[Dict]:
    """
    Returns dict with per-eye 64×64 crops, or None if no face found.

    Crop geometry mirrors streamlit_app_v7.py inference exactly:
        IOD = |le_x - re_x|
        per-eye width  = IOD * CFG.iod_width_ratio
        per-eye height = IOD * CFG.iod_height_ratio

    YuNet landmark layout (15 values per face):
      [0]=x [1]=y [2]=w [3]=h
      [4]=re_x [5]=re_y   (right eye centre)
      [6]=le_x [7]=le_y   (left  eye centre)
      [8]=nose_x [9]=nose_y
      [10..13] = mouth corners
      [14]=score
    """
    if frame_bgr is None:
        return None
    h, w = frame_bgr.shape[:2]
    try:
        detector.setInputSize((w, h))
        _, faces = detector.detect(frame_bgr)
    except Exception:
        return None
    if faces is None or len(faces) == 0:
        return None

    det = faces[0]
    score = float(det[14]) if len(det) > 14 else 0.0
    re_x, re_y = float(det[4]), float(det[5])
    le_x, le_y = float(det[6]), float(det[7])
    iod = abs(le_x - re_x)
    if iod < 8:        # too small / degenerate detection
        return None

    half_w = iod * CFG.iod_width_ratio  / 2.0
    half_h = iod * CFG.iod_height_ratio / 2.0

    def _crop(cx, cy):
        x1 = max(0, int(cx - half_w))
        x2 = min(w, int(cx + half_w))
        y1 = max(0, int(cy - half_h))
        y2 = min(h, int(cy + half_h))
        if x2 <= x1 or y2 <= y1:
            return None
        patch = frame_bgr[y1:y2, x1:x2]
        return cv2.resize(patch, (CFG.img_size, CFG.img_size),
                          interpolation=cv2.INTER_LINEAR)

    right = _crop(re_x, re_y)
    left  = _crop(le_x, le_y)

    if right is None and left is None:
        return None

    return {"right": right, "left": left, "iod": iod, "score": score}


# Download YuNet now
ensure_dir(os.path.dirname(CFG.yunet_model_path))
download_yunet(CFG.yunet_model_path)
print("YuNet ready.")

YuNet already present: /kaggle/working/face_detection_yunet_2023mar.onnx
YuNet ready.


In [31]:
# ══════════════════════════════════════════════════════════════════════════════
# EYE CROP CACHE  (NTHU frames)
# Precomputes per-eye crops from NTHU frames using YuNet IOD geometry.
# Caches to disk as 64×64 JPEG files so training is fast.
# Each NTHU frame → up to 2 cache entries: {img_path}_L.jpg  {img_path}_R.jpg
# ══════════════════════════════════════════════════════════════════════════════

class NTHUEyeCropCache:
    """
    Manages precomputed per-eye crops from NTHU frames.
    Falls back to face-region heuristic if YuNet fails.
    """
    def __init__(self, cache_root: str, yunet_path: str, rebuild: bool = False):
        self.cache_root = cache_root
        ensure_dir(cache_root)

        self._detector = make_yunet_detector(yunet_path)

        # index maps image_path → {"L": path_or_None, "R": path_or_None}
        self._index_path = os.path.join(cache_root, "index.json")
        if rebuild and os.path.exists(self._index_path):
            os.remove(self._index_path)
        self._index: Dict[str, Dict] = {}
        if os.path.exists(self._index_path):
            with open(self._index_path) as f:
                self._index = json.load(f)

    def _cache_key(self, img_path: str, side: str) -> str:
        h = abs(hash(img_path)) % (10**8)
        return os.path.join(self.cache_root, f"{h}_{side}.jpg")

    def _process_one(self, img_path: str) -> Dict:
        """Returns {L: path_or_None, R: path_or_None}."""
        result = {"L": None, "R": None}
        frame = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if frame is None:
            return result

        # Apply lighting normalisation before detection (matches inference)
        frame, _ = normalize_lighting(frame)

        eyes = yunet_detect_eyes(frame, self._detector)

        if eyes is None:
            # Fallback: use upper-middle region of frame as approximate eye strip,
            # split in half for left/right
            h, w = frame.shape[:2]
            eye_region = frame[int(h*0.20):int(h*0.55), int(w*0.10):int(w*0.90)]
            if eye_region.size == 0:
                return result
            mid = eye_region.shape[1] // 2
            right_half = eye_region[:, :mid]
            left_half  = eye_region[:, mid:]
            eyes = {"right": cv2.resize(right_half, (CFG.img_size, CFG.img_size)),
                    "left":  cv2.resize(left_half,  (CFG.img_size, CFG.img_size)),
                    "iod": 0.0, "score": 0.0}

        for side, crop in [("R", eyes.get("right")), ("L", eyes.get("left"))]:
            if crop is not None and crop.size > 0:
                dst = self._cache_key(img_path, side)
                cv2.imwrite(dst, crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
                result[side] = dst

        return result

    def get(self, img_path: str) -> Dict:
        """Returns cached crop paths, computing on cache miss."""
        if img_path in self._index:
            return self._index[img_path]
        result = self._process_one(img_path)
        self._index[img_path] = result
        return result

    def precompute(self, image_paths: List[str], save_every: int = 2000):
        """Batch precompute all paths not yet in cache."""
        todo = [p for p in image_paths if p not in self._index]
        print(f"[Cache] {len(image_paths)} total | {len(todo)} to process")
        for i, p in enumerate(todo):
            self.get(p)
            if (i + 1) % save_every == 0:
                self._save_index()
                print(f"  Cached {i+1}/{len(todo)} …")
        self._save_index()
        print("[Cache] Precompute complete.")

    def _save_index(self):
        with open(self._index_path, "w") as f:
            json.dump(self._index, f)

print("NTHUEyeCropCache defined.")

NTHUEyeCropCache defined.


In [32]:
# ══════════════════════════════════════════════════════════════════════════════
# EYE LABEL PARSING  (NTHU)
# Unchanged from v2 — this logic is correct.
# ══════════════════════════════════════════════════════════════════════════════

def parse_a2_eye_label(filename: str, class_name: str = "",
                        rel_path: str = "",
                        main_label: Optional[int] = None) -> int:
    """
    Returns:
      1  = eye closed (positive)
      0  = eye open   (negative)
     -1  = ambiguous  (ignore / masked out in loss)
    """
    text = compact_text(filename, class_name, rel_path)

    explicit_close_tokens = ["closeeye", "eyeclose"]
    slowblink_tokens      = ["slowblink", "slowblinkwithnoding"]
    explicit_neg_tokens   = ["nonsleepycombination", "normal"]
    ambiguous_tokens      = ["sleepycombination", "yawn", "noding", "nodding",
                              "headnod", "drinking", "smoking", "talking",
                              "phone", "texting", "looking", "laugh"]

    has_close    = any(tok in text for tok in explicit_close_tokens)
    has_slow     = any(tok in text for tok in slowblink_tokens)
    has_neg      = any(tok in text for tok in explicit_neg_tokens)
    has_ambig    = any(tok in text for tok in ambiguous_tokens)
    has_notdrowsy = "notdrowsy" in text

    if has_close: return 1
    if has_slow and main_label == 1: return 1
    if has_neg: return 0
    if has_notdrowsy and not has_ambig and not has_slow: return 0
    return -1


print("parse_a2_eye_label() defined.")

parse_a2_eye_label() defined.


In [33]:
# ══════════════════════════════════════════════════════════════════════════════
# NTHU MANIFEST LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_nthu_fold(manifest_root: str, fold_id: int,
                    data_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Loads train/val/test CSVs for one NTHU fold."""
    fold_dir = os.path.join(manifest_root, "nthu", f"fold_{fold_id}")
    splits = {}
    for split in ("train", "val", "test"):
        csv_path = os.path.join(fold_dir, f"{split}.csv")
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Missing: {csv_path}")
        df = pd.read_csv(csv_path)
        df["image_path"] = df["path"].apply(
            lambda x: normalize_manifest_path(x, data_dir))
        df["filename"] = df["image_path"].apply(os.path.basename)

        if "class_name" not in df.columns: df["class_name"] = ""

        if "subject_id" not in df.columns:
            df["subject_id"] = (df["filename"]
                                .str.extract(r"^(\d+)")[0]
                                .fillna(-1).astype(int))
        else:
            df["subject_id"] = df["subject_id"].fillna(-1).astype(int)

        if "label" not in df.columns:
            df["label"] = (df["class_name"].str.lower()
                           .map({"notdrowsy": 0, "drowsy": 1}))
        df["label"] = df["label"].astype(int)

        df["eye_label"] = [
            parse_a2_eye_label(fn, cls, rel, ml)
            for fn, cls, rel, ml in zip(
                df["filename"], df["class_name"].astype(str),
                df["path"].astype(str), df["label"])
        ]

        # Drop images that don't exist on disk
        exists = df["image_path"].apply(os.path.exists)
        dropped = (~exists).sum()
        if dropped: print(f"[WARN] {split}: {dropped} missing files dropped.")
        df = df[exists].reset_index(drop=True)
        splits[split] = df

    # Verify no subject leakage
    train_subj = set(splits["train"]["subject_id"].unique())
    val_subj   = set(splits["val"]  ["subject_id"].unique())
    test_subj  = set(splits["test"] ["subject_id"].unique())
    assert not (train_subj & val_subj),  f"Subject leakage train/val:  {train_subj & val_subj}"
    assert not (train_subj & test_subj), f"Subject leakage train/test: {train_subj & test_subj}"

    return splits["train"], splits["val"], splits["test"]


print("load_nthu_fold() defined.")

load_nthu_fold() defined.


In [34]:
# ══════════════════════════════════════════════════════════════════════════════
# MRL EYE DATASET LOADER
#
# MRL filename format: s{subj}_{imgid}_{light}_{glasses}_{eye_state}_{refl}_{lc}_{sensor}.png
# Position [4] in underscore-split: 0 = CLOSED, 1 = OPEN
# Example: s0001_02694_0_0_1_0_0_01.png  → eye_state=1 → OPEN
#
# Kaggle dataset: tauilabdelilah/mrl-eye-dataset
# Structure: mrlEyes_2018_01/s0001/ ... s0037/ → *.png files
# ══════════════════════════════════════════════════════════════════════════════

def scan_mrl_dataset(mrl_dir: str,
                      max_samples: int = 8000,
                      seed: int = 42) -> pd.DataFrame:
    """
    Scans MRL dataset. Decodes label from filename position 4:
      '0' = closed (positive), '1' = open (negative).
    Returns DataFrame with [image_path, eye_label].
    """
    if not os.path.isdir(mrl_dir):
        print(f"[MRL] Directory not found: {mrl_dir} — skipping.")
        return pd.DataFrame(columns=["image_path", "eye_label"])

    rows = []
    skipped = 0
    for root, _, files in os.walk(mrl_dir):
        for fname in files:
            if not fname.lower().endswith(".png"):
                continue
            parts = fname.replace(".png", "").split("_")
            # MRL naming: s####_#####_light_glasses_eye_state_refl_lc_sensor
            if len(parts) < 8:
                # Some MRL mirrors use different naming — try fallback
                ftext = fname.lower()
                if "closed" in ftext or "close" in ftext:
                    label = 1
                elif "open" in ftext:
                    label = 0
                else:
                    skipped += 1
                    continue
            else:
                eye_state_str = parts[4]   # '0' = closed, '1' = open
                if eye_state_str == "0":
                    label = 1              # closed = positive
                elif eye_state_str == "1":
                    label = 0              # open   = negative
                else:
                    skipped += 1
                    continue
            rows.append({"image_path": os.path.join(root, fname),
                          "eye_label":  label})

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"[MRL] No valid images found in {mrl_dir}.  skipped={skipped}")
        return df

    raw_counts = df["eye_label"].value_counts().to_dict()
    print(f"[MRL] Raw: {len(df)} images  {raw_counts}  (skipped={skipped})")

    # Balanced subsample
    half      = max_samples // 2
    open_df   = df[df["eye_label"] == 0]
    closed_df = df[df["eye_label"] == 1]
    n_open    = min(half, len(open_df))
    n_closed  = min(half, len(closed_df))
    sampled   = pd.concat([
        open_df.sample(  n=n_open,   random_state=seed, replace=False),
        closed_df.sample(n=n_closed, random_state=seed, replace=False),
    ]).reset_index(drop=True)

    print(f"[MRL] Sampled: {len(sampled)} (open={n_open}, closed={n_closed})")
    return sampled


# ══════════════════════════════════════════════════════════════════════════════
# EXTERNAL EYE DATASET LOADER (replaces CEW)
# dhirdevansh/eye-dataset-openclose-for-drowsiness-prediction
#
# Structure: open/  and  closed/  subfolders containing eye patch images.
# Falls back to filename text matching if folder names differ.
# ══════════════════════════════════════════════════════════════════════════════

def scan_ext_eye_dataset(ext_dir: str,
                          max_samples: int = 2000,
                          seed: int = 42) -> pd.DataFrame:
    """
    Scans folder-based eye dataset.
    Label from directory name: 'closed' / 'close' → 1 (closed), 'open' → 0 (open).
    Returns DataFrame with [image_path, eye_label].
    """
    if not os.path.isdir(ext_dir):
        print(f"[EXT] Directory not found: {ext_dir} — skipping.")
        return pd.DataFrame(columns=["image_path", "eye_label"])

    rows = []
    for root, dirs, files in os.walk(ext_dir):
        dir_lower = os.path.basename(root).lower()
        if "closed" in dir_lower or "close" in dir_lower:
            label = 1
        elif "open" in dir_lower:
            label = 0
        else:
            continue
        for fname in files:
            if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                rows.append({"image_path": os.path.join(root, fname),
                              "eye_label":  label})

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"[EXT] No images found in {ext_dir}.")
        return df

    raw_counts = df["eye_label"].value_counts().to_dict()
    print(f"[EXT] Raw: {len(df)} images  {raw_counts}")

    half      = max_samples // 2
    open_df   = df[df["eye_label"] == 0]
    closed_df = df[df["eye_label"] == 1]
    n_open    = min(half, len(open_df))
    n_closed  = min(half, len(closed_df))
    sampled   = pd.concat([
        open_df.sample(  n=n_open,   random_state=seed, replace=False),
        closed_df.sample(n=n_closed, random_state=seed, replace=False),
    ]).reset_index(drop=True)

    print(f"[EXT] Sampled: {len(sampled)} (open={n_open}, closed={n_closed})")
    return sampled


print("MRL / external eye dataset loaders defined.")
print("MRL label decoding: filename.split('_')[4] → '0'=closed → label=1, '1'=open → label=0")


MRL / external eye dataset loaders defined.
MRL label decoding: filename.split('_')[4] → '0'=closed → label=1, '1'=open → label=0


In [35]:
# ══════════════════════════════════════════════════════════════════════════════
# AUGMENTATION PIPELINE
# Training augmentations aggressively simulate real-world conditions:
#   • lighting variation (CLAHE, gamma, brightness, contrast)
#   • partial occlusion (glasses frames, eyelashes)
#   • motion blur, noise
#   • horizontal flip (eyes are roughly symmetric)
# Validation / test: deterministic resize + normalise only.
# ══════════════════════════════════════════════════════════════════════════════

def get_train_transforms(img_size: int = 64) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),

        # ── Lighting variation (critical for day/night/tunnel) ────────────────
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.35, contrast_limit=0.35, p=0.8),
            A.RandomGamma(gamma_limit=(60, 140), p=0.8),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(4, 4), p=0.8),
        ], p=0.85),

        # ── Simulate IR / grayscale cameras ──────────────────────────────────
        A.ToGray(p=0.15),

        # ── Geometric: small translate + scale (not big rotations for eyes) ──
        A.Affine(
            scale=(0.85, 1.15),
            translate_percent={"x": (-0.08, 0.08), "y": (-0.08, 0.08)},
            rotate=(-10, 10),
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.55,
        ),

        # ── Horizontal flip: eyes are roughly symmetric ───────────────────────
        A.HorizontalFlip(p=0.50),

        # ── Partial occlusion: glasses frame, eyelash shadows, hair ──────────
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.08, 0.20),
            hole_width_range=(0.10, 0.30),
            fill_value=0, p=0.30,
        ),

        # ── Blur / noise: camera shake, compression ───────────────────────────
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 5), p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.5),
        ], p=0.25),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.10), p=0.6),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=0.4),
        ], p=0.25),

        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size: int = 64) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


print("Augmentation pipelines defined.")

Augmentation pipelines defined.


In [36]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET CLASSES
# ══════════════════════════════════════════════════════════════════════════════

class NTHUPerEyeDataset(Dataset):
    """
    Each NTHU frame produces up to 2 samples: left-eye and right-eye crops.
    crop_cache must be a NTHUEyeCropCache instance.
    Only frames with eye_label != -1 and at least one valid crop are included.
    """
    def __init__(self, df: pd.DataFrame, crop_cache: NTHUEyeCropCache,
                 transform: A.Compose, name: str = ""):
        self.transform = transform
        rows = []
        for _, row in df.iterrows():
            lbl = int(row["eye_label"])
            if lbl == -1:
                continue
            crops = crop_cache.get(str(row["image_path"]))
            for side in ("L", "R"):
                cp = crops.get(side)
                if cp and os.path.exists(cp):
                    rows.append({"crop_path": cp, "eye_label": lbl})
        self._rows = rows
        lbl_counts = pd.Series([r["eye_label"] for r in rows]).value_counts().to_dict()
        print(f"[NTHUPerEye/{name}] {len(rows)} samples  {lbl_counts}")

    def __len__(self): return len(self._rows)

    def __getitem__(self, idx):
        r   = self._rows[idx]
        img = cv2.imread(r["crop_path"], cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        aug = self.transform(image=img)
        return aug["image"], torch.tensor(float(r["eye_label"]), dtype=torch.float32)


class ExternalEyeDataset(Dataset):
    """
    Generic dataset for MRL / CEW — images are already individual eye crops.
    Loads image, resizes/normalises via albumentations transform.
    """
    def __init__(self, df: pd.DataFrame, transform: A.Compose, name: str = ""):
        self.transform = transform
        self._rows = df[["image_path", "eye_label"]].to_dict("records")
        lbl_counts = df["eye_label"].value_counts().to_dict()
        print(f"[ExternalEye/{name}] {len(self._rows)} samples  {lbl_counts}")

    def __len__(self): return len(self._rows)

    def __getitem__(self, idx):
        r   = self._rows[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        elif img.ndim == 2:          # grayscale → 3-channel
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        aug = self.transform(image=img)
        return aug["image"], torch.tensor(float(r["eye_label"]), dtype=torch.float32)


from torch.utils.data import ConcatDataset

print("Dataset classes defined.")

Dataset classes defined.


In [37]:
# ══════════════════════════════════════════════════════════════════════════════
# CBAM — Channel and Spatial Attention
# Added between backbone features and the classification head.
# Helps the model focus on the pupil / iris region rather than skin / hair.
# ══════════════════════════════════════════════════════════════════════════════

class ChannelAttention(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 8):
        super().__init__()
        mid = max(1, in_channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # x: (B, C) — already pooled
        return x * self.fc(x)


class CBAM1D(nn.Module):
    """Lightweight 1-D CBAM for post-GAP features (B, C)."""
    def __init__(self, in_channels: int, reduction: int = 8):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction)

    def forward(self, x):
        return self.ca(x)


print("CBAM1D defined.")

CBAM1D defined.


In [38]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════════════

class EyeStateModel(nn.Module):
    """
    MobileNetV3-Small backbone + optional CBAM attention + sigmoid head.
    Input:  (B, 3, 64, 64) — per-eye crop.
    Output: (B,)  raw logit (apply sigmoid for probability).

    At inference (Snowflake CPU):
      - Run on left-eye crop  → p_left
      - Run on right-eye crop → p_right
      - eye_closed_prob = (p_left + p_right) / 2
    """
    def __init__(self, backbone_name: str = "mobilenetv3_small_100",
                 pretrained: bool = True,
                 dropout: float = 0.30,
                 use_cbam: bool = True,
                 img_size: int = 64):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            num_classes=0, global_pool="avg")

        # Infer feature dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            feats = self.backbone(dummy)
            if feats.ndim > 2: feats = feats.flatten(1)
            feat_dim = feats.shape[1]

        self.use_cbam = use_cbam
        if use_cbam:
            self.cbam = CBAM1D(feat_dim, reduction=8)

        self.dropout = nn.Dropout(p=dropout)
        self.head    = nn.Linear(feat_dim, 1)

        print(f"EyeStateModel: {backbone_name}  feat_dim={feat_dim}  "
              f"cbam={use_cbam}  dropout={dropout}")

    def forward(self, x):
        feats = self.backbone(x)
        if feats.ndim > 2: feats = feats.flatten(1)
        if self.use_cbam: feats = self.cbam(feats)
        feats = self.dropout(feats)
        return self.head(feats).squeeze(1)   # raw logit, shape (B,)


print("EyeStateModel defined.")

EyeStateModel defined.


In [39]:
# ══════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS
# Reduces the relative loss for well-classified easy (open-eye) examples.
# Forces the model to focus training on hard borderline cases.
# ══════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """
    Binary Focal Loss.  Inputs are raw logits (not probabilities).
        FL(p) = -alpha * (1-p)^gamma * log(p)  for positive class
              = -(1-alpha) * p^gamma * log(1-p) for negative class
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce  = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none")
        p    = torch.sigmoid(logits)
        p_t  = p * targets + (1 - p) * (1 - targets)
        a_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = a_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


def build_criterion(use_focal: bool, pos_weight=None, device=None):
    if use_focal:
        print(f"Loss: FocalLoss(gamma={CFG.focal_gamma}, alpha={CFG.focal_alpha})")
        return FocalLoss(gamma=CFG.focal_gamma, alpha=CFG.focal_alpha).to(device)
    pw = torch.tensor([pos_weight], dtype=torch.float32).to(device) if pos_weight else None
    print(f"Loss: BCE  pos_weight={pos_weight}")
    return nn.BCEWithLogitsLoss(pos_weight=pw).to(device)


print("FocalLoss defined.")

FocalLoss defined.


In [40]:
# ══════════════════════════════════════════════════════════════════════════════
# BALANCED SAMPLER
# ══════════════════════════════════════════════════════════════════════════════

def build_balanced_sampler(labels) -> WeightedRandomSampler:
    """
    labels: list or array of 0/1 int labels.
    Returns a WeightedRandomSampler that gives equal probability to open/closed.
    """
    labels = np.asarray(labels, dtype=int)
    counts = np.bincount(labels, minlength=2)
    weights = np.zeros(2, dtype=np.float64)
    for cls in [0, 1]:
        if counts[cls] > 0:
            weights[cls] = 1.0 / counts[cls]
    sample_weights = torch.as_tensor(weights[labels], dtype=torch.double)
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


def get_labels_from_concat(dataset) -> List[int]:
    """Extract eye_label list from a ConcatDataset."""
    labels = []
    for ds in dataset.datasets:
        if isinstance(ds, NTHUPerEyeDataset):
            labels.extend(r["eye_label"] for r in ds._rows)
        elif isinstance(ds, ExternalEyeDataset):
            labels.extend(r["eye_label"] for r in ds._rows)
    return labels


print("Balanced sampler defined.")

Balanced sampler defined.


In [41]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAIN / EVALUATE
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, scaler, device, criterion,
                     use_amp=True, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, targets)
        scaler.scale(loss).backward()
        if grad_clip > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return {"loss": total_loss / max(1, len(loader))}


@torch.no_grad()
def evaluate(model, loader, device, criterion, use_amp=True):
    model.eval()
    all_logits, all_targets = [], []
    total_loss = 0.0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        with autocast(enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, targets)
        total_loss += loss.item()
        all_logits.extend(logits.cpu().numpy().tolist())
        all_targets.extend(targets.cpu().numpy().tolist())

    probs   = sigmoid_np(all_logits)
    targets = np.asarray(all_targets)
    auc     = safe_auc(targets, probs)
    ap      = safe_ap(targets, probs)
    best_thr, best_f1 = find_best_threshold(targets, probs, CFG.threshold_steps)

    return {
        "loss":           total_loss / max(1, len(loader)),
        "auc":            auc,
        "ap":             ap,
        "best_threshold": best_thr,
        "best_f1":        best_f1,
    }


print("train_one_epoch / evaluate defined.")

train_one_epoch / evaluate defined.


In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# RUN ONE FOLD
# ══════════════════════════════════════════════════════════════════════════════

def run_fold(fold_id: int, mrl_df: pd.DataFrame, ext_df: pd.DataFrame):
    seed_everything(CFG.seed, CFG.deterministic, CFG.cudnn_benchmark)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"FOLD {fold_id}   device={device}")
    print(f"{'='*60}")

    fold_dir = os.path.join(CFG.output_root, f"fold_{fold_id}")
    ensure_dir(fold_dir)

    # ── Load NTHU manifests ───────────────────────────────────────────────────
    train_df, val_df, test_df = load_nthu_fold(
        CFG.manifest_root, fold_id, CFG.nthu_data_dir)
    print(f"NTHU  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

    # ── Precompute eye crops ──────────────────────────────────────────────────
    cache = NTHUEyeCropCache(
        cache_root=os.path.join(CFG.crop_cache_root, f"fold_{fold_id}"),
        yunet_path=CFG.yunet_model_path,
        rebuild=CFG.rebuild_crop_cache,
    )
    all_paths = (
        train_df["image_path"].tolist() +
        val_df  ["image_path"].tolist() +
        test_df ["image_path"].tolist()
    )
    cache.precompute(all_paths, save_every=CFG.save_cache_every)

    # ── Transforms ───────────────────────────────────────────────────────────
    train_tfm = get_train_transforms(CFG.img_size)
    val_tfm   = get_val_transforms(CFG.img_size)

    # ── Datasets ─────────────────────────────────────────────────────────────
    nthu_train_ds = NTHUPerEyeDataset(train_df, cache, train_tfm, name=f"fold{fold_id}/train")
    nthu_val_ds   = NTHUPerEyeDataset(val_df,   cache, val_tfm,   name=f"fold{fold_id}/val")
    nthu_test_ds  = NTHUPerEyeDataset(test_df,  cache, val_tfm,   name=f"fold{fold_id}/test")

    # Validate label counts
    if len(nthu_val_ds) < CFG.min_known_samples_per_split:
        print(f"[WARN] Fold {fold_id}: val only {len(nthu_val_ds)} samples.")

    # Combine NTHU train + external datasets
    train_parts = [nthu_train_ds]
    if not mrl_df.empty:
        train_parts.append(ExternalEyeDataset(
            mrl_df, train_tfm, name=f"MRL/fold{fold_id}"))
    if not ext_df.empty:
        train_parts.append(ExternalEyeDataset(
            ext_df, train_tfm, name=f"EXT/fold{fold_id}"))

    if len(train_parts) > 1:
        train_ds = ConcatDataset(train_parts)
        print(f"Combined train dataset: {len(train_ds)} samples")
    else:
        train_ds = nthu_train_ds

    # ── Sampler ──────────────────────────────────────────────────────────────
    sampler = None
    shuffle = True
    if CFG.use_balanced_sampler:
        if len(train_parts) > 1:
            all_labels = get_labels_from_concat(train_ds)
        else:
            all_labels = [r["eye_label"] for r in nthu_train_ds._rows]
        sampler = build_balanced_sampler(all_labels)
        shuffle = False   # WeightedRandomSampler handles randomisation
        print(f"Balanced sampler: {len(all_labels)} samples")

    loader_g = torch.Generator(); loader_g.manual_seed(CFG.seed)
    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, shuffle=shuffle,
        sampler=sampler, num_workers=CFG.num_workers, pin_memory=True,
        drop_last=True, worker_init_fn=seed_worker, generator=loader_g)
    val_loader  = DataLoader(
        nthu_val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True)
    test_loader = DataLoader(
        nthu_test_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True)

    # ── Model, optimiser, scheduler ──────────────────────────────────────────
    model = EyeStateModel(
        backbone_name=CFG.backbone, pretrained=True,
        dropout=CFG.dropout, use_cbam=CFG.use_cbam,
        img_size=CFG.img_size,
    ).to(device)

    criterion = build_criterion(CFG.use_focal_loss, device=device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.epochs, eta_min=1e-6)
    scaler = GradScaler(enabled=CFG.use_amp and device.type == "cuda")

    # ── SWA setup ─────────────────────────────────────────────────────────────
    swa_model    = AveragedModel(model) if CFG.use_swa else None
    swa_start_ep = int(CFG.swa_start * CFG.epochs)
    swa_scheduler = (SWALR(optimizer, swa_lr=CFG.swa_lr,
                            anneal_epochs=5, anneal_strategy="cos")
                     if CFG.use_swa else None)
    if CFG.use_swa:
        print(f"SWA activates at epoch {swa_start_ep}  swa_lr={CFG.swa_lr}")

    # ── Training loop ─────────────────────────────────────────────────────────
    best_score     = -np.inf
    best_threshold = 0.5
    best_epoch     = 0
    no_improve     = 0
    history        = []
    best_path      = os.path.join(fold_dir, f"a2_fold_{fold_id}_best.pt")
    last_path      = os.path.join(fold_dir, "last.pt")

    for epoch in range(1, CFG.epochs + 1):
        print(f"\n--- Epoch {epoch}/{CFG.epochs} ---")

        train_m = train_one_epoch(
            model, train_loader, optimizer, scaler, device, criterion,
            use_amp=CFG.use_amp and device.type == "cuda",
            grad_clip=CFG.grad_clip)

        val_m = evaluate(
            model, val_loader, device, criterion,
            use_amp=CFG.use_amp and device.type == "cuda")

        # SWA update
        if CFG.use_swa and epoch >= swa_start_ep:
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_m.items()},
               **{f"val_{k}": v for k, v in val_m.items()}}
        history.append(row)
        pd.DataFrame(history).to_csv(
            os.path.join(fold_dir, "history.csv"), index=False)

        # Primary metric: AUC, fallback to best_f1
        score = val_m["auc"] if not math.isnan(val_m["auc"]) else val_m["best_f1"]
        print({"train_loss": round(train_m["loss"], 4),
               "val_loss":   round(val_m["loss"], 4),
               "val_auc":    round(val_m["auc"], 4) if not math.isnan(val_m["auc"]) else "nan",
               "val_ap":     round(val_m["ap"],  4) if not math.isnan(val_m["ap"])  else "nan",
               "best_thr":   round(val_m["best_threshold"], 4),
               "best_f1":    round(val_m["best_f1"], 4)})

        # Checkpoint
        state = {
            "epoch": epoch,
            "model_state_dict":  model.state_dict(),
            "cfg":               asdict(CFG),
            "fold_id":           fold_id,
            "best_score":        best_score,
            "best_eye_threshold": best_threshold,
            "best_epoch":        best_epoch,
        }
        torch.save(state, last_path)

        if score > best_score + 1e-6:
            best_score     = score
            best_threshold = float(val_m["best_threshold"])
            best_epoch     = epoch
            no_improve     = 0
            state["best_score"]         = best_score
            state["best_eye_threshold"] = best_threshold
            state["best_epoch"]         = best_epoch
            torch.save(state, best_path)
            print(f"[INFO] New best  AUC={best_score:.4f}  thr={best_threshold:.4f}")
        else:
            no_improve += 1

        if epoch >= CFG.min_epochs and no_improve >= CFG.patience:
            print(f"[INFO] Early stopping at epoch {epoch}.")
            break

        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()

    # ── SWA: update BatchNorm stats, then evaluate ────────────────────────────
    if CFG.use_swa:
        print("\nUpdating SWA BatchNorm stats …")
        update_bn(train_loader, swa_model, device=device)
        swa_val_m = evaluate(
            swa_model, val_loader, device, criterion,
            use_amp=False)   # SWA model on CPU-compatible path
        print("SWA val metrics:", {k: round(v, 4) if not math.isnan(v) else "nan"
                                     for k, v in swa_val_m.items()})
        if swa_val_m["auc"] > best_score:
            best_score     = swa_val_m["auc"]
            best_threshold = float(swa_val_m["best_threshold"])
            print(f"[SWA] Better: AUC={best_score:.4f}  thr={best_threshold:.4f}")
            torch.save({
                "epoch":             best_epoch,
                "model_state_dict":  swa_model.module.state_dict(),
                "cfg":               asdict(CFG),
                "fold_id":           fold_id,
                "best_score":        best_score,
                "best_eye_threshold": best_threshold,
                "best_epoch":        best_epoch,
                "source":            "swa",
            }, best_path)

    # ── Final test evaluation ─────────────────────────────────────────────────
    print("\nLoading best checkpoint for test evaluation …")
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    best_eval_thr = float(ckpt.get("best_eye_threshold", 0.5))

    test_m = evaluate(
        model, test_loader, device, criterion,
        use_amp=CFG.use_amp and device.type == "cuda")
    print(f"\nTest @ best_val_thr={best_eval_thr:.4f}:")
    print({k: round(v, 4) if not math.isnan(v) else "nan"
           for k, v in test_m.items()})

    summary = {
        "fold_id":              fold_id,
        "best_epoch":           best_epoch,
        "best_val_auc":         best_score,
        "best_val_threshold":   best_threshold,
        "test_auc":             test_m["auc"],
        "test_ap":              test_m["ap"],
        "test_best_f1":         test_m["best_f1"],
        "test_best_threshold":  test_m["best_threshold"],
        "checkpoint_path":      best_path,
    }
    with open(os.path.join(fold_dir, "fold_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nFold {fold_id} complete. Summary saved to {fold_dir}")
    return summary


print("run_fold() defined.")

run_fold() defined.


In [43]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN — Run all folds
# ══════════════════════════════════════════════════════════════════════════════

ensure_dir(CFG.output_root)

# ── Load external datasets once (shared across all folds) ─────────────────────
print("\n=== Loading MRL Eye Dataset ===")
mrl_df = scan_mrl_dataset(
    CFG.mrl_dir, max_samples=CFG.mrl_samples_per_fold, seed=CFG.seed)

print("\n=== Loading External Eye Dataset (dhirdevansh) ===")
ext_df = scan_ext_eye_dataset(
    CFG.ext_dir, max_samples=CFG.ext_samples_per_fold, seed=CFG.seed)

# ── Run folds ─────────────────────────────────────────────────────────────────
all_summaries = []
for fold_id in CFG.fold_ids:
    summary = run_fold(fold_id, mrl_df, ext_df)
    all_summaries.append(summary)

print("\n" + "="*60)
print("ALL FOLDS COMPLETE")
print("="*60)
for s in all_summaries:
    print(f"  Fold {s['fold_id']}:  val_auc={s['best_val_auc']:.4f}  "
          f"test_auc={s['test_auc']:.4f}  "
          f"best_thr={s['best_val_threshold']:.4f}")


=== Loading MRL Eye Dataset ===
[MRL] Raw: 84898 images  {0: 42952, 1: 41946}  (skipped=0)
[MRL] Sampled: 8000 (open=4000, closed=4000)

=== Loading External Eye Dataset (dhirdevansh) ===
[EXT] Raw: 4000 images  {0: 2000, 1: 2000}
[EXT] Sampled: 2000 (open=1000, closed=1000)

FOLD 1   device=cuda
NTHU  train=28672  val=18833  test=19016
[Cache] 66521 total | 0 to process
[Cache] Precompute complete.
[NTHUPerEye/fold1/train] 26750 samples  {0: 18486, 1: 8264}
[NTHUPerEye/fold1/val] 17044 samples  {0: 11226, 1: 5818}
[NTHUPerEye/fold1/test] 16866 samples  {0: 12124, 1: 4742}
[ExternalEye/MRL/fold1] 8000 samples  {0: 4000, 1: 4000}
[ExternalEye/EXT/fold1] 2000 samples  {0: 1000, 1: 1000}
Combined train dataset: 36750 samples
Balanced sampler: 36750 samples
EyeStateModel: mobilenetv3_small_100  feat_dim=1024  cbam=True  dropout=0.3
Loss: FocalLoss(gamma=2.0, alpha=0.75)
SWA activates at epoch 8  swa_lr=5e-05

--- Epoch 1/25 ---
{'train_loss': 0.0361, 'val_loss': 0.094, 'val_auc': 0.7012, 

In [44]:
import shutil

folder_path = "/kaggle/working/eye_crop_cache_v3/fold_3"

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print("Folder and all contents deleted")
else:
    print("Folder not found")

Folder and all contents deleted


In [45]:
# ══════════════════════════════════════════════════════════════════════════════
# ENSEMBLE META  —  pick best 3 folds, compute mean threshold
# Mirrors the structure of a2_ensemble_meta.json used by the app.
# ══════════════════════════════════════════════════════════════════════════════

# Sort folds by test AUC, take top 3
sorted_folds = sorted(
    all_summaries,
    key=lambda s: s["test_auc"] if not math.isnan(s["test_auc"]) else -1,
    reverse=True,
)
top3 = sorted_folds[:3]
fold_ids_selected = [s["fold_id"] for s in top3]
thresholds        = [s["best_val_threshold"] for s in top3]
mean_threshold    = float(np.mean(thresholds))

meta = {
    "version":          "a2_v3_mobilenetv3_small_per_eye_64x64",
    "backbone":         CFG.backbone,
    "img_size":         CFG.img_size,
    "use_cbam":         CFG.use_cbam,
    "crop_geometry":    "yunet_iod_based",
    "iod_width_ratio":  CFG.iod_width_ratio,
    "iod_height_ratio": CFG.iod_height_ratio,
    "folds_selected":   fold_ids_selected,
    "fold_files":       [f"a2_fold_{f}_best.pt" for f in fold_ids_selected],
    "fold_thresholds":  thresholds,
    "mean_threshold":   mean_threshold,
    "focal_gamma":      CFG.focal_gamma,
    "focal_alpha":      CFG.focal_alpha,
    "fold_summaries":   all_summaries,
}

meta_path = os.path.join(CFG.output_root, "a2_ensemble_meta_v3.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("\n=== ENSEMBLE META ===")
print(f"Selected folds:    {fold_ids_selected}")
print(f"Per-fold thresholds: {[round(t,4) for t in thresholds]}")
print(f"Mean threshold:    {mean_threshold:.4f}")
print(f"Saved to:          {meta_path}")
print("\nDeploy the following files to @DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS:")
for fold_file in meta["fold_files"]:
    fold_n = fold_file.replace("a2_fold_", "").replace("_best.pt", "")
    src = os.path.join(CFG.output_root, f"fold_{fold_n}", fold_file)
    print(f"  {src}")


=== ENSEMBLE META ===
Selected folds:    [3, 2, 1]
Per-fold thresholds: [0.302, 0.3119, 0.5049]
Mean threshold:    0.3730
Saved to:          /kaggle/working/model_a2_v3/a2_ensemble_meta_v3.json

Deploy the following files to @DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS:
  /kaggle/working/model_a2_v3/fold_3/a2_fold_3_best.pt
  /kaggle/working/model_a2_v3/fold_2/a2_fold_2_best.pt
  /kaggle/working/model_a2_v3/fold_1/a2_fold_1_best.pt


In [46]:
# ══════════════════════════════════════════════════════════════════════════════
# COPY OUTPUT FILES TO /kaggle/working ROOT (for easy download)
# ══════════════════════════════════════════════════════════════════════════════
import shutil

for s in all_summaries:
    fold_n = s["fold_id"]
    src    = os.path.join(CFG.output_root, f"fold_{fold_n}",
                          f"a2_fold_{fold_n}_best.pt")
    dst    = f"/kaggle/working/a2_fold_{fold_n}_best.pt"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied fold {fold_n} → {dst}")

shutil.copy2(meta_path, "/kaggle/working/a2_ensemble_meta_v3.json")
print("\nAll output files are in /kaggle/working — ready to download.")
print("Upload to Snowflake stage:\n"
      "  PUT file:///path/a2_fold_*_best.pt @DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS AUTO_COMPRESS=FALSE;")

Copied fold 1 → /kaggle/working/a2_fold_1_best.pt
Copied fold 2 → /kaggle/working/a2_fold_2_best.pt
Copied fold 3 → /kaggle/working/a2_fold_3_best.pt
Copied fold 4 → /kaggle/working/a2_fold_4_best.pt

All output files are in /kaggle/working — ready to download.
Upload to Snowflake stage:
  PUT file:///path/a2_fold_*_best.pt @DEMO_DB.PUBLIC.DRIVER_SAFETY_MODELS AUTO_COMPRESS=FALSE;
